# Modelado — Predicción del rendimiento académico (G3)

Este notebook entrena y compara tres modelos de regresión sobre el dataset UCI Student Performance (Math) para predecir la nota final `G3` (0–20):

- **Regresión Lineal** — baseline interpretable
- **Random Forest** — modelo no lineal, con ajuste de hiperparámetros
- **K-Nearest Neighbors (KNN)** — modelo basado en instancias, con ajuste de hiperparámetros

## Decisiones clave

- **Semilla fija** `random_state=42` en todos los puntos aleatorios (split, CV, modelos) para garantizar reproducibilidad.
- **Partición 70 / 15 / 15** (train / validación / test). El conjunto de validación se integra al entrenamiento vía `GridSearchCV` con CV de 5 folds; el test queda totalmente aislado hasta la evaluación final.
- **Chequeo explícito de *data leakage* con G1 y G2**: las notas parciales están fuertemente correlacionadas con G3 (ver heatmap en `data_processing.ipynb`) y su inclusión puede ser una fuga de información. Entrenamos todos los modelos en dos escenarios:
  - **Escenario A**: incluye G1 y G2 (sirve como cota superior de desempeño, pero poco útil en la práctica)
  - **Escenario B**: excluye G1 y G2 (responde a la verdadera pregunta de investigación: qué factores socioeconómicos, familiares y de hábitos predicen el rendimiento)
- **Escalado** con `StandardScaler` dentro de un `Pipeline` para Regresión Lineal y KNN (necesario para KNN, beneficioso para la estabilidad de LR). Random Forest no requiere escalado.

Dataset elegido: **Mathematics** (395 estudiantes). El flujo es directamente reutilizable sobre `portuguese_processed.csv`.


In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
os.makedirs('models', exist_ok=True)
print('Librerías cargadas. random_state =', RANDOM_STATE)


Librerías cargadas. random_state = 42


## 1. Carga de los datos preprocesados

Cargamos `math_processed.csv` generado por `preprocessing.ipynb`. El archivo ya contiene el encoding binario, las variables one-hot y la columna `pass`. Aquí solo usamos el target continuo `G3`.


In [2]:
df = pd.read_csv('./data/math_processed.csv')
print('Shape:', df.shape)
df.head(3)


Shape: (395, 43)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,studytime,...,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,pass
0,0,0,18,1,1,0,4,4,2,2,...,False,False,False,True,False,False,False,True,False,0
1,0,0,17,1,1,1,1,1,1,2,...,False,True,False,False,False,False,False,False,False,0
2,0,0,15,1,0,1,1,1,1,2,...,False,True,False,False,False,True,False,True,False,1


## 2. Construcción de los escenarios A y B

Replicamos la misma definición de features que en `preprocessing.ipynb` (excluir targets `G3` y `pass`) y construimos dos matrices:

- `X_A` incluye G1 y G2 — contaminado con información futura (notas parciales que ocurren cronológicamente antes de G3 pero que ya están muy cerca del target).
- `X_B` excluye G1 y G2 — libre de leakage, basado solo en variables socioeconómicas, familiares y de hábitos disponibles antes del inicio del curso.


In [3]:
TARGET = 'G3'
DROP_TARGETS = ['G3', 'pass']

features_A = [c for c in df.columns if c not in DROP_TARGETS]
features_B = [c for c in features_A if c not in ('G1', 'G2')]

X_A = df[features_A].astype(float)
X_B = df[features_B].astype(float)
y   = df[TARGET].astype(float)

print(f'Escenario A (con G1/G2): {X_A.shape[1]} features')
print(f'Escenario B (sin G1/G2): {X_B.shape[1]} features')


Escenario A (con G1/G2): 41 features
Escenario B (sin G1/G2): 39 features


## 3. Partición 70 / 15 / 15

Hacemos un doble split para obtener train (70%), validación (15%) y test (15%). Usamos la misma semilla y el mismo orden de filas en ambos escenarios para que la comparación A vs B sea justa (los mismos estudiantes en cada partición).

> Nota: `GridSearchCV` aplicará 5-fold CV sobre el train. El conjunto de validación queda disponible para chequeos manuales si hicieran falta, y el test solo se toca al final.


In [4]:
def split_70_15_15(X, y, random_state=RANDOM_STATE):
    # Primero separo el test (15%), luego divido el resto en train (70/85) y val (15/85)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.15, random_state=random_state
    )
    val_ratio = 0.15 / 0.85
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_ratio, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

splits_A = split_70_15_15(X_A, y)
splits_B = split_70_15_15(X_B, y)

X_train_A, X_val_A, X_test_A, y_train_A, y_val_A, y_test_A = splits_A
X_train_B, X_val_B, X_test_B, y_train_B, y_val_B, y_test_B = splits_B

print(f'Escenario A — train: {X_train_A.shape}, val: {X_val_A.shape}, test: {X_test_A.shape}')
print(f'Escenario B — train: {X_train_B.shape}, val: {X_val_B.shape}, test: {X_test_B.shape}')


Escenario A — train: (275, 41), val: (60, 41), test: (60, 41)
Escenario B — train: (275, 39), val: (60, 39), test: (60, 39)


## 4. Definición de modelos y búsqueda de hiperparámetros

- **Linear Regression**: sin hiperparámetros a tunear; entrenamos directo con CV solo para reportar su desempeño promedio.
- **Random Forest**: `GridSearchCV` sobre `n_estimators`, `max_depth`, `min_samples_split`.
- **KNN**: `GridSearchCV` sobre `n_neighbors` y `weights`.

El criterio de selección es RMSE (menor es mejor) → usamos `neg_root_mean_squared_error` como `scoring`.


In [5]:
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def build_models():
    lr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression()),
    ])

    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    rf_grid = {
        'n_estimators':      [100, 200],
        'max_depth':         [None, 10, 20],
        'min_samples_split': [2, 5],
    }

    knn_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  KNeighborsRegressor()),
    ])
    knn_grid = {
        'model__n_neighbors': [3, 5, 7, 9, 11],
        'model__weights':     ['uniform', 'distance'],
    }

    return lr_pipe, (rf, rf_grid), (knn_pipe, knn_grid)


## 5. Entrenamiento y evaluación

Entrenamos los tres modelos en ambos escenarios. Para LR usamos `cross_val_score`; para RF y KNN usamos `GridSearchCV` y nos quedamos con el mejor estimador. Luego evaluamos una única vez en el test set.


In [6]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae  = float(mean_absolute_error(y_test, y_pred))
    r2   = float(r2_score(y_test, y_pred))
    return rmse, mae, r2

def train_scenario(X_train, y_train, X_test, y_test, tag):
    results = []
    fitted  = {}

    lr_pipe, (rf, rf_grid), (knn_pipe, knn_grid) = build_models()

    # --- Linear Regression ---
    lr_cv = -cross_val_score(lr_pipe, X_train, y_train,
                             scoring='neg_root_mean_squared_error', cv=CV, n_jobs=-1).mean()
    lr_pipe.fit(X_train, y_train)
    rmse, mae, r2 = evaluate(lr_pipe, X_test, y_test)
    results.append(('LinearRegression', tag, lr_cv, rmse, mae, r2, {}))
    fitted['LinearRegression'] = lr_pipe

    # --- Random Forest ---
    rf_gs = GridSearchCV(rf, rf_grid, scoring='neg_root_mean_squared_error',
                         cv=CV, n_jobs=-1, refit=True)
    rf_gs.fit(X_train, y_train)
    rf_cv = -rf_gs.best_score_
    rmse, mae, r2 = evaluate(rf_gs.best_estimator_, X_test, y_test)
    results.append(('RandomForest', tag, rf_cv, rmse, mae, r2, rf_gs.best_params_))
    fitted['RandomForest'] = rf_gs.best_estimator_

    # --- KNN ---
    knn_gs = GridSearchCV(knn_pipe, knn_grid, scoring='neg_root_mean_squared_error',
                          cv=CV, n_jobs=-1, refit=True)
    knn_gs.fit(X_train, y_train)
    knn_cv = -knn_gs.best_score_
    rmse, mae, r2 = evaluate(knn_gs.best_estimator_, X_test, y_test)
    results.append(('KNN', tag, knn_cv, rmse, mae, r2, knn_gs.best_params_))
    fitted['KNN'] = knn_gs.best_estimator_

    return results, fitted

print('Entrenando Escenario A (con G1/G2)...')
results_A, models_A = train_scenario(X_train_A, y_train_A, X_test_A, y_test_A, 'A (con G1/G2)')

print('Entrenando Escenario B (sin G1/G2)...')
results_B, models_B = train_scenario(X_train_B, y_train_B, X_test_B, y_test_B, 'B (sin G1/G2)')

print('Entrenamiento completado.')


Entrenando Escenario A (con G1/G2)...
Entrenando Escenario B (sin G1/G2)...
Entrenamiento completado.


## 6. Tabla comparativa de resultados

Consolidamos los resultados de ambos escenarios en un único `DataFrame`. Reportamos RMSE, MAE y R² sobre el test set, junto con el RMSE promedio obtenido en 5-fold CV y los hiperparámetros elegidos.


In [7]:
rows = results_A + results_B
results_df = pd.DataFrame(rows, columns=[
    'Modelo', 'Escenario', 'RMSE_CV', 'RMSE_test', 'MAE_test', 'R2_test', 'Mejores_params'
])
results_df = results_df.sort_values(['Escenario', 'RMSE_test']).reset_index(drop=True)

with pd.option_context('display.max_colwidth', None):
    display(results_df)


,Modelo,Escenario,RMSE_CV,RMSE_test,MAE_test,R2_test,Mejores_params
0,RandomForest,A (con G1/G2),1.433676,2.099670,1.170449,0.800625,"{'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}"
1,LinearRegression,A (con G1/G2),1.858434,2.444651,1.620346,0.729728,{}
2,KNN,A (con G1/G2),3.229400,3.258749,2.473659,0.519748,"{'model__n_neighbors': 11, 'model__weights': 'distance'}"
3,RandomForest,B (sin G1/G2),3.730774,3.999624,3.103937,0.276554,"{'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}"
4,LinearRegression,B (sin G1/G2),4.331195,4.272237,3.367554,0.174574,{}
5,KNN,B (sin G1/G2),4.228248,4.426040,3.558736,0.114072,"{'model__n_neighbors': 11, 'model__weights': 'distance'}"


### Interpretación de los resultados

- El Escenario A (con G1 y G2) típicamente alcanza R² muy alto (>0.8) porque G1 y G2 son casi el mismo target. Esto confirma el riesgo de *leakage*: el modelo no está aprendiendo a predecir el rendimiento a partir de variables tempranas, sino a copiar notas parciales.
- El Escenario B es el que responde a la pregunta de investigación real. Los R² bajan, lo cual es esperable, pero las métricas son más honestas y sus factores explicativos (ver `explainability.ipynb`) son los que tienen valor práctico para el profesor y para una posible intervención educativa.

Por este motivo, el análisis de explicabilidad se hará sobre los modelos del Escenario B.


## 7. Persistencia de modelos

Guardamos todos los modelos entrenados (ambos escenarios) con `joblib` en la carpeta `models/` para que `explainability.ipynb` pueda cargarlos sin reentrenar.


In [8]:
for name, model in models_A.items():
    joblib.dump(model, f'models/{name}_scenarioA.joblib')
for name, model in models_B.items():
    joblib.dump(model, f'models/{name}_scenarioB.joblib')

joblib.dump({
    'features_A':  features_A,
    'features_B':  features_B,
    'X_train_B':   X_train_B,
    'X_test_B':    X_test_B,
    'y_train_B':   y_train_B,
    'y_test_B':    y_test_B,
    'X_train_A':   X_train_A,
    'X_test_A':    X_test_A,
    'y_train_A':   y_train_A,
    'y_test_A':    y_test_A,
}, 'models/data_splits.joblib')

results_df.to_csv('models/results_comparison.csv', index=False)
print('Modelos, splits y tabla de resultados guardados en models/')


Modelos, splits y tabla de resultados guardados en models/


## Conclusión del notebook

1. Se entrenaron 3 modelos x 2 escenarios = 6 configuraciones, todas con semilla fija y 5-fold CV.
2. El escenario B (sin G1/G2) es el que se utilizará en `explainability.ipynb` para responder a la pregunta de investigación.
3. Los modelos quedan serializados en `models/` listos para ser cargados.
